In [1]:
import cudf
from cuml.cluster import KMeans, DBSCAN
from cuml.svm import OneClassSVM
from cuml.ensemble import IsolationForest
from cuml.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

def train_test_anomaly_detection_rapids(train_csv_path, test_csv_path, model_type='ocsvm', model_params=None):
    # Load and preprocess data using cuDF
    train_data = cudf.read_csv(train_csv_path)
    test_data = cudf.read_csv(test_csv_path)
    train_data = train_data.fillna(0)
    test_data = test_data.fillna(0)

    # Extract features and labels from test data
    X_train = train_data.drop(columns=['Label'])
    X_test = test_data.drop(columns=['Label'])
    y_test = test_data['Label'].to_array()  # Convert to NumPy array for evaluation

    # Initialize the model
    if model_type == 'ocsvm':
        model = OneClassSVM(**(model_params or {}))
    elif model_type == 'iforest':
        model = IsolationForest(**(model_params or {}))
    elif model_type == 'lof':
        model = LocalOutlierFactor(**(model_params or {}))
    elif model_type == 'kmeans':
        model = KMeans(**(model_params or {}))
    elif model_type == 'dbscan':
        model = DBSCAN(**(model_params or {}))
    else:
        raise ValueError("Unsupported model type")

    # Train and predict
    model.fit(X_train)
    y_pred_test = model.predict(X_test)

    # Adjust predictions based on the model type
    if model_type in ['ocsvm', 'iforest', 'dbscan']:
        y_pred_test = [1 if pred == -1 else 0 for pred in y_pred_test.to_array()]
    elif model_type == 'kmeans':
        # For KMeans, determine the threshold for anomalies
        # Example: Using a distance threshold
        centroids = model.cluster_centers_
        distances = cudf.DataFrame()
        for i, centroid in enumerate(centroids):
            col = f'cluster_{i}'
            distances[col] = ((X_test - centroid) ** 2).sum(axis=1).sqrt()
        min_distance = distances.min(axis=1)
        threshold = min_distance.quantile(0.95)  # Example threshold
        y_pred_test = [1 if dist > threshold else 0 for dist in min_distance.to_array()]
    elif model_type == 'lof':
        # For LOF, negative_outlier_factor_ can be used to determine anomalies
        # Example: Using a LOF score threshold
        threshold = -1.5  # Example threshold
        y_pred_test = [1 if lof < threshold else 0 for lof in model.negative_outlier_factor_.to_array()]

    # Evaluate the model
    print("Classification Report:\n", classification_report(y_test, y_pred_test))
    print("Accuracy Score:", accuracy_score(y_test, y_pred_test))
    print("Precision Score:", precision_score(y_test, y_pred_test))
    print("Recall Score:", recall_score(y_test, y_pred_test))
    print("F1 Score:", f1_score(y_test, y_pred_test))

    # Confusion matrix for additional metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
    print("True Positive Rate (Sensitivity):", tp / (tp + fn))
    print("False Positive Rate:", fp / (fp + tn))
    print("True Negative Rate (Specificity):", tn / (tn + fp))
    print("Positive Predictive Value (Precision):", tp / (tp + fp))
    print("Negative Predictive Value:", tn / (tn + fn))

# Example usage
# train_test_anomaly_detection_rapids('path_to_train_csv.csv', 'path_to_test_csv.csv', model_type='ocsvm', model_params={'gamma': 'auto', 'nu': 0.05})
train_test_anomaly_detection(
    '../data_selected/syncan/train.csv', 
    '../data_selected/syncan/test_flooding.csv', 
    model_type='ocsvm', 
    model_params={'gamma': 'auto', 'nu': 0.05})


ImportError: cannot import name 'OneClassSVM' from 'cuml.svm' (/home/ubuntu/anaconda3/envs/CANGen/lib/python3.9/site-packages/cuml/svm/__init__.py)